# 🦅 Set Up

''' To use GPU
conda install -y pytorch pytorch-cuda=12.1 -c pytorch -c nvidia
'''

In [1]:
from functions import *

In [2]:
cd ..

C:\Users\chopi\Penn Dropbox\Hyunwoo Jung\1_Personal\_Hyunwoo Place\graduate school\2_coursework (2025-F)\2_CIS5200_Machine Learning\5_final project\3_analyses


# 🦅 Prep Data
- Use the same features and data set

In [3]:
df_sg = pd.read_parquet("data/appliances_stage0_v1.parquet")

In [4]:
df_sg.head().iloc[:, :20]

,review_id,rating,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,text_len,text_words,main_category,average_rating,rating_number,price,store,categories,MPN_1,MPN_2,MPN_3,MPN_4
0,6,5.0,B00004YWK2,B00004YWK2,AFNA7RNBEH66UUMHPEE7XJFB3MYA,2023-01-03 12:37:29.824,1,True,175,33,Tools & Home Improvement,4.1,579,NaN,Dundas Jafine,Parts & Accessories,0.037446,0.011010,0.022477,0.038484
1,5,5.0,B00004YWK2,B00004YWK2,AHKTIX6L7FKPYDENUALEPNPMAIHQ,2023-01-03 16:32:51.697,0,True,708,148,Tools & Home Improvement,4.1,579,NaN,Dundas Jafine,Parts & Accessories,0.003132,-0.005602,0.004761,-0.021004
2,4,5.0,B00004YWK2,B00004YWK2,AGO4SBTXOUTKYMHKQQNX7ZFDQSFA,2023-01-29 19:37:40.587,0,True,115,24,Tools & Home Improvement,4.1,579,NaN,Dundas Jafine,Parts & Accessories,-0.069358,-0.000819,0.006222,0.020757
3,3,1.0,B00004YWK2,B00004YWK2,AGA5X6NUWSQM42KDFJLO25JBXOEA,2023-02-07 14:55:58.766,0,True,47,9,Tools & Home Improvement,4.1,579,NaN,Dundas Jafine,Parts & Accessories,0.031263,0.027401,0.002945,0.014352
4,2,1.0,B00004YWK2,B00004YWK2,AEI6B25VF65CG2HPBQ2FNBG7IQKA,2023-02-10 00:31:39.499,0,True,47,9,Tools & Home Improvement,4.1,579,NaN,Dundas Jafine,Parts & Accessories,0.054404,0.025682,0.012061,0.018683


In [5]:
df_with_sae = pd.read_feather('data/appliances_reviews_012023_062023 v1.4.0 (SAE embedding).ftr')

In [6]:
df_with_sae.head().iloc[:, :10]

,review_id,item_id,text,inter_review_time,irt_days,y_log_irt,sae_0,sae_1,sae_2,sae_3
0,2,B00004YWK2,Product is cheap and the door doesn't work well,8 days 02:00:26.779000,8.083643,2.206475,0.000000,1.425354,1.305985,0.000000
1,3,B00004YWK2,No cover to close it and plastic was separated.,2 days 09:35:40.733000,2.399777,1.223710,0.224967,1.231542,1.122654,0.101840
2,4,B00004YWK2,"I love this it keeps my garage, nice and warm ...",8 days 19:18:18.179000,8.804377,2.282829,0.226200,1.067915,1.358308,0.102711
3,5,B00004YWK2,"So I mounted this, as you see, behind and just...",26 days 03:04:48.890000,26.128344,3.300579,0.663035,1.612317,1.635834,0.299319
4,6,B00004YWK2,I have 3 if these and the 1st bought several y...,0 days 03:55:21.873000,0.163448,0.151388,0.477116,1.353411,1.241897,0.195181


In [7]:
df_sg.shape, df_with_sae.shape

((94297, 1306), (94297, 3078))

In [8]:
df_sg_mrgSAE = df_sg[['review_id', 'timestamp', 'rating', 'helpful_vote', 'verified_purchase', 'text_len', 'text_words', 'average_rating', 'rating_number', 'y_hours', 'y_log', 'fold']].merge(
    df_with_sae.drop(columns=['inter_review_time', 'irt_days', 'y_log_irt']), how='left', on='review_id'
)

In [9]:
df_sg_mrgSAE.head().iloc[:, :20]

,review_id,timestamp,rating,helpful_vote,verified_purchase,text_len,text_words,average_rating,rating_number,y_hours,y_log,fold,item_id,text,sae_0,sae_1,sae_2,sae_3,sae_4,sae_5
0,6,2023-01-03 12:37:29.824,5.0,1,True,175,33,4.1,579,3.922743,1.593866,train,B00004YWK2,I have 3 if these and the 1st bought several y...,0.477116,1.353411,1.241897,0.195181,0.0,1.211486
1,5,2023-01-03 16:32:51.697,5.0,0,True,708,148,4.1,579,627.080247,6.442668,train,B00004YWK2,"So I mounted this, as you see, behind and just...",0.663035,1.612317,1.635834,0.299319,0.0,1.360516
2,4,2023-01-29 19:37:40.587,5.0,0,True,115,24,4.1,579,211.305050,5.358024,train,B00004YWK2,"I love this it keeps my garage, nice and warm ...",0.226200,1.067915,1.358308,0.102711,0.0,1.161993
3,3,2023-02-07 14:55:58.766,1.0,0,True,47,9,4.1,579,57.594648,4.070643,train,B00004YWK2,No cover to close it and plastic was separated.,0.224967,1.231542,1.122654,0.101840,0.0,0.752137
4,2,2023-02-10 00:31:39.499,1.0,0,True,47,9,4.1,579,194.007439,5.273038,train,B00004YWK2,Product is cheap and the door doesn't work well,0.000000,1.425354,1.305985,0.000000,0.0,0.901288


In [12]:
df_sg_mrgSAE.to_feather('data/appliances_reviews_012023_062023 v1.4.0 (SAE embedding, meta).ftr')

# 🦅 Load Data

In [3]:
df_model = pd.read_feather('data/appliances_reviews_012023_062023 v1.4.0 (SAE embedding, meta).ftr')
df_model = df_model[df_model['sae_0'].notnull()]

In [4]:
df_model['verified_purchase'] = df_model['verified_purchase'].replace({True:1, False:0})

C:\Users\chopi\AppData\Local\Temp\ipykernel_1979160\3091700506.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_model['verified_purchase'] = df_model['verified_purchase'].replace({True:1, False:0})


In [5]:
df_model.head().iloc[:, :20]

,review_id,timestamp,rating,helpful_vote,verified_purchase,text_len,text_words,average_rating,rating_number,y_hours,y_log,fold,item_id,text,sae_0,sae_1,sae_2,sae_3,sae_4,sae_5
0,6,2023-01-03 12:37:29.824,5.0,1,1,175,33,4.1,579,3.922743,1.593866,train,B00004YWK2,I have 3 if these and the 1st bought several y...,0.477116,1.353411,1.241897,0.195181,0.0,1.211486
1,5,2023-01-03 16:32:51.697,5.0,0,1,708,148,4.1,579,627.080247,6.442668,train,B00004YWK2,"So I mounted this, as you see, behind and just...",0.663035,1.612317,1.635834,0.299319,0.0,1.360516
2,4,2023-01-29 19:37:40.587,5.0,0,1,115,24,4.1,579,211.305050,5.358024,train,B00004YWK2,"I love this it keeps my garage, nice and warm ...",0.226200,1.067915,1.358308,0.102711,0.0,1.161993
3,3,2023-02-07 14:55:58.766,1.0,0,1,47,9,4.1,579,57.594648,4.070643,train,B00004YWK2,No cover to close it and plastic was separated.,0.224967,1.231542,1.122654,0.101840,0.0,0.752137
4,2,2023-02-10 00:31:39.499,1.0,0,1,47,9,4.1,579,194.007439,5.273038,train,B00004YWK2,Product is cheap and the door doesn't work well,0.000000,1.425354,1.305985,0.000000,0.0,0.901288


# 🦅 Prediction

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error

import lightgbm as lgb

In [7]:
''' Feature definition '''
metadata_cols = ['rating', 'helpful_vote', 'verified_purchase', 'text_len', 'text_words', 'average_rating', 'rating_number']
sae_cols = [c for c in df_model.columns if c.startswith('sae_')]
feature_cols = metadata_cols + sae_cols

target_col = 'y_log'

In [8]:
''' Train / Validation / Test split '''
df_train_valid = df_model[df_model['fold'].isin(['train', 'valid'])].copy()
df_test = df_model[df_model['fold'] == 'test'].copy()

X_train_valid = df_train_valid[feature_cols].copy()
y_train_valid = df_train_valid[target_col].copy()

X_test = df_test[feature_cols].copy()
y_test = df_test[target_col].copy()

In [9]:
fold_map = {'train': -1, 'valid': 0}
test_fold = df_train_valid['fold'].map(fold_map).values

ps = PredefinedSplit(test_fold=test_fold)

In [10]:
del df_model, df_train_valid, df_test

## 🐔 Ridge

In [10]:
ridge = Ridge(random_state=42)

param_grid_ridge = {
    'alpha': [0.1, 0.3, 1.0, 3.0, 10.0, 30.0]
}

ridge_grid = GridSearchCV(
    estimator=ridge,
    param_grid=param_grid_ridge,
    scoring='neg_root_mean_squared_error',  # tuning on RMSE
    cv=ps,
    n_jobs=-1,
    refit=True,
    verbose=1
)

ridge_grid.fit(X_train_valid, y_train_valid)

best_ridge = ridge_grid.best_estimator_
print("Best Ridge params:", ridge_grid.best_params_)

Fitting 1 folds for each of 6 candidates, totalling 6 fits
Best Ridge params: {'alpha': 30.0}


In [12]:
# Evaluate on test
y_pred_ridge_test = best_ridge.predict(X_test)
ridge_rmse_test = mean_squared_error(y_test, y_pred_ridge_test)
ridge_mae_test = mean_absolute_error(y_test, y_pred_ridge_test)

print(f"[Ridge] Test RMSE: {ridge_rmse_test:.4f}")
print(f"[Ridge] Test MAE : {ridge_mae_test:.4f}")

[Ridge] Test RMSE: 2.6436
[Ridge] Test MAE : 1.2162


## 🐔 LGBM

In [11]:
lgb_reg = lgb.LGBMRegressor(
    objective='regression',
    random_state=42,
    n_jobs=1,
    #device='gpu'   # requires LightGBM with GPU support
)

param_grid_lgb = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.001, 0.0001],
    'max_depth': [10, 20, 30]
}

lgb_grid = GridSearchCV(
    estimator=lgb_reg,
    param_grid=param_grid_lgb,
    scoring='neg_root_mean_squared_error',  # tuning on RMSE
    cv=ps,
    n_jobs=1,
    refit=True,
    verbose=1
)

lgb_grid.fit(X_train_valid, y_train_valid)

best_lgb = lgb_grid.best_estimator_
print("Best LightGBM params:", lgb_grid.best_params_)

Fitting 1 folds for each of 27 candidates, totalling 27 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 3.229659 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765529
[LightGBM] [Info] Number of data points in the train set: 85229, number of used features: 3071
[LightGBM] [Info] Start training from score 4.218596
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 3.185914 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765529
[LightGBM] [Info] Number of data points in the train set: 85229, number of used features: 3071
[LightGBM] [Info] Start training from score 4.218596
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 3.244994 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765529
[LightGBM] [Info] Number of data points in the tr

In [13]:
# Evaluate on test
y_pred_lgb_test = best_lgb.predict(X_test)
lgb_rmse_test = mean_squared_error(y_test, y_pred_lgb_test)
lgb_mae_test = mean_absolute_error(y_test, y_pred_lgb_test)

print(f"[LightGBM] Test RMSE: {lgb_rmse_test:.4f}")
print(f"[LightGBM] Test MAE : {lgb_mae_test:.4f}")

[LightGBM] Test RMSE: 2.3718
[LightGBM] Test MAE : 1.1720


# 🦅 Inspection

In [3]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error

import lightgbm as lgb

In [4]:
df_model = pd.read_feather('data/appliances_reviews_012023_062023 v1.4.0 (SAE embedding, meta).ftr')

df_model = df_model[df_model['sae_0'].notnull()]
df_model['verified_purchase'] = df_model['verified_purchase'].replace({True:1, False:0})

C:\Users\chopi\AppData\Local\Temp\ipykernel_21384\4087134500.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_model['verified_purchase'] = df_model['verified_purchase'].replace({True:1, False:0})


In [5]:
''' Feature definition '''
metadata_cols = ['rating', 'helpful_vote', 'verified_purchase', 'text_len', 'text_words', 'average_rating', 'rating_number']
sae_cols = [c for c in df_model.columns if c.startswith('sae_')]
feature_cols = metadata_cols + sae_cols

target_col = 'y_log'

In [6]:
''' Train / Validation / Test split '''
df_train_valid = df_model[df_model['fold'].isin(['train', 'valid'])].copy()
df_test = df_model[df_model['fold'] == 'test'].copy()

X_train_valid = df_train_valid[feature_cols].copy()
y_train_valid = df_train_valid[target_col].copy()

X_test = df_test[feature_cols].copy()
y_test = df_test[target_col].copy()

In [7]:
fold_map = {'train': -1, 'valid': 0}
test_fold = df_train_valid['fold'].map(fold_map).values

ps = PredefinedSplit(test_fold=test_fold)

In [8]:
del df_model, df_test

## 🐔 best model

In [9]:
lgb_reg = lgb.LGBMRegressor(
    objective='regression',
    random_state=42,
    n_jobs=1,
    n_estimators=100,
    max_depth=10,
    learning_rate=0.01,
)

lgb_reg.fit(X_train_valid, y_train_valid)
y_pred_lgb_test = lgb_reg.predict(X_test)

lgb_rmse_test = mean_squared_error(y_test, y_pred_lgb_test)
lgb_mae_test = mean_absolute_error(y_test, y_pred_lgb_test)

print(f"[LightGBM] Test RMSE: {lgb_rmse_test:.4f}")
print(f"[LightGBM] Test MAE : {lgb_mae_test:.4f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 3.516685 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 766172
[LightGBM] [Info] Number of data points in the train set: 91495, number of used features: 3071
[LightGBM] [Info] Start training from score 4.239545
[LightGBM] Test RMSE: 2.3718
[LightGBM] Test MAE : 1.1720


In [10]:
# 1) SHAP values on validation set
explainer = shap.TreeExplainer(lgb_reg)
sv = explainer.shap_values(X_train_valid)           # shape: [n_samples, n_features]
mean_abs = np.abs(sv).mean(axis=0)                  # magnitude (importance)
mean_signed = sv.mean(axis=0)                       # direction on average (±)

order = np.argsort(mean_abs)[::-1]
topK = min(20, len(order))
sae_cols = [f"sae_f{i}" for i in range(lgb_reg.n_features_in_)]

print("Top features by |SHAP| (validation):")
for idx in order[:topK]:
    direction = "↑ IRT (lower demand)" if mean_signed[idx] > 0 else (
                "↓ IRT (higher demand)" if mean_signed[idx] < 0 else "neutral")
    print(f"{idx:>5}  {sae_cols[idx]:<40}  |shap|={mean_abs[idx]:.5f}  mean_shap={mean_signed[idx]:+.5f}  {direction}")

Top features by |SHAP| (validation):
    6  sae_f6                                    |shap|=0.50365  mean_shap=+0.00000  ↑ IRT (lower demand)
    2  sae_f2                                    |shap|=0.09407  mean_shap=-0.00371  ↓ IRT (higher demand)
    5  sae_f5                                    |shap|=0.08164  mean_shap=-0.00239  ↓ IRT (higher demand)
    1  sae_f1                                    |shap|=0.01561  mean_shap=+0.00052  ↑ IRT (lower demand)
 1051  sae_f1051                                 |shap|=0.00643  mean_shap=-0.00011  ↓ IRT (higher demand)
  324  sae_f324                                  |shap|=0.00402  mean_shap=+0.00063  ↑ IRT (lower demand)
 1892  sae_f1892                                 |shap|=0.00383  mean_shap=-0.00007  ↓ IRT (higher demand)
    0  sae_f0                                    |shap|=0.00370  mean_shap=+0.00093  ↑ IRT (lower demand)
 2294  sae_f2294                                 |shap|=0.00363  mean_shap=+0.00040  ↑ IRT (lower demand)
 1119

In [12]:
# Frozen LM
lm = FrozenLM(LMConfig(model_name="roberta-base", layer_index=10))
sae, meta = load_sae("data/model/sae_robertaL10_k3072_1115_2025_113219.pt", device=lm.cfg.device)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\chopi\Penn Dropbox\Hyunwoo Jung\1_Personal\_Hyunwoo Place\graduate school\2_coursework (2025-F)\2_CIS5200_Machine Learning\5_final project\3_analyses\jupyter\functions.py:533: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allow

### higher demand

In [34]:
inspect_sae_feature(feature_idx=2, lm=lm, sae=sae, F_va=df_train_valid[[c for c in df_train_valid.columns if c.startswith('sae_')]].values, texts_va=df_train_valid['text'].values, order=order, n_reviews_to_show=10)


Inspecting SAE Feature: 2

--- Top Example 1 (Review Index: 87147) ---
Review-Level Feature Activation: 2.3302

Highlighted Review (Feature Activation Points in RED):
Great washer my only issue is as you can tell by the picture the drain hose of the picture that they put up shows it's on the right side but it's not it's on the left so I had to face my washer the opposite way never seen one like that or had one like that before but it's a fabulous washer I've done multiple loads cleans them very well just like a normal big washer would I love it the only thing I would change like I said is the drain hose on the opposite side


Activating Spans Found:
  - (Score: 2.4403) @ [226:230]: ' like'
------------------------------------------------------------

--- Top Example 2 (Review Index: 25299) ---
Review-Level Feature Activation: 2.2865

Highlighted Review (Feature Activation Points in RED):
I was so excited about this machine. I love the ice that it produces. However, I don’t know that t

In [35]:
inspect_sae_feature(feature_idx=5, lm=lm, sae=sae, F_va=df_train_valid[[c for c in df_train_valid.columns if c.startswith('sae_')]].values, texts_va=df_train_valid['text'].values, order=order, n_reviews_to_show=10)


Inspecting SAE Feature: 5

--- Top Example 1 (Review Index: 78872) ---
Review-Level Feature Activation: 1.9397

Highlighted Review (Feature Activation Points in RED):
I was hoping they'd stick better and be a bit more rigid. My stove back is pretty slim so if I put anything heavy close enough to the back the whole thing will fall behind my stove. My stove is also gas so I can't push it back up against the wall, which would solve my problem. So only get this if your stove ridge is 2-3" wide and/or you have an electric stove. Otherwise you'll need to find something to support this shelf besides the internal magnets.


Activating Spans Found:
  - (Score: 2.1151) @ [46:50]: ' more'
------------------------------------------------------------

--- Top Example 2 (Review Index: 26261) ---
Review-Level Feature Activation: 1.9047

Highlighted Review (Feature Activation Points in RED):
Its okay, it keeps the solids off the surface so scrubbing is kept to a miniumum i suppose. Is it worth 40$? T

In [36]:
inspect_sae_feature(feature_idx=1051, lm=lm, sae=sae, F_va=df_train_valid[[c for c in df_train_valid.columns if c.startswith('sae_')]].values, texts_va=df_train_valid['text'].values, order=order, n_reviews_to_show=10)


Inspecting SAE Feature: 1051

--- Top Example 1 (Review Index: 82594) ---
Review-Level Feature Activation: 1.9061

Highlighted Review (Feature Activation Points in RED):
This bullet ice maker generally works as advertised.  Setup is a breeze add water, plug it in, push the clean button, drain it after cleaning, fill it again with water and walk away.  It produces 9 ice cubes every 8 minutes +or- 15 seconds.  When the bin is full it stops making ice and the ice full light comes on.  Take out some ice and it starts making ice again.  The same is true with the water level, stops when low, the add water light comes on and ice making resumes after adding water.  When the ice bin is full it holds 1.4 lbs of ice.<br />The only complaint I have is you need to remove the ice bin to refill the water reservoir, not a deal breaker but a bit of a pain.


Activating Spans Found:
  - (Score: 1.9698) @ [127:132]: ' after'
------------------------------------------------------------

--- Top Example 2

In [37]:
inspect_sae_feature(feature_idx=1892, lm=lm, sae=sae, F_va=df_train_valid[[c for c in df_train_valid.columns if c.startswith('sae_')]].values, texts_va=df_train_valid['text'].values, order=order, n_reviews_to_show=10)


Inspecting SAE Feature: 1892

--- Top Example 1 (Review Index: 42410) ---
Review-Level Feature Activation: 0.4384

Highlighted Review (Feature Activation Points in RED):
𝚆𝚘𝚛𝚔𝚜 𝚕𝚒𝚔𝚎 𝚊 𝚌𝚑𝚊𝚖𝚙!  𝙾𝚗𝚌𝚎 𝚒𝚝 𝚛𝚞𝚗𝚜 𝚒𝚝'𝚜 𝚏𝚒𝚛𝚜𝚝 𝚌𝚢𝚌𝚕𝚎 𝚒𝚝 𝚜𝚝𝚊𝚛𝚝𝚜 𝚌𝚛𝚊𝚗𝚔𝚒𝚗𝚐 𝚘𝚞𝚝 𝚝𝚑𝚎 𝚒𝚌𝚎.  𝙹𝚞𝚜𝚝 𝚊𝚍𝚓𝚞𝚜𝚝 𝚝𝚑𝚎 𝚝𝚒𝚖𝚎 𝚏𝚘𝚛 𝚢𝚘𝚞𝚛 𝚙𝚛𝚎𝚏𝚎𝚛𝚛𝚎𝚍 𝚝𝚑𝚒𝚌𝚔𝚗𝚎𝚜𝚜.


Activating Spans Found:
  - (Score: 0.4720) @ [14:15]: '�'
------------------------------------------------------------

--- Top Example 2 (Review Index: 70932) ---
Review-Level Feature Activation: 0.2679

Highlighted Review (Feature Activation Points in RED):
Purchased this product one week ago for occasional use.  Used 2 of the pods so far and went to fill the 3rd pod and when I opened the lid, there were coffee grounds all of over the inside of the lid, as well as three holes on the bottom of the pod.  It's clear that someone returned this product after using and breaking it.  Considering this is a tool used for FOOD product, it is absolutely unsanitary for a company to resell to 

In [38]:
inspect_sae_feature(feature_idx=2327, lm=lm, sae=sae, F_va=df_train_valid[[c for c in df_train_valid.columns if c.startswith('sae_')]].values, texts_va=df_train_valid['text'].values, order=order, n_reviews_to_show=10)


Inspecting SAE Feature: 2327

--- Top Example 1 (Review Index: 7443) ---
Review-Level Feature Activation: 2.2463

Highlighted Review (Feature Activation Points in RED):
We have a parts per million measurement tool in our house to measure the water to make sure whatever water filter we purchase actually is doing what it claims to do.<br />We took our tap water measurement, 166 parts per million. We installed the filters and ran 16 cups thru it as recommended then measured.  22 parts per million. Good.  Checked absinthe the next morning,  we're up to 133 parts per million.<br />These are complete junk


Activating Spans Found:
  - (Score: 2.5906) @ [433:437]: ' junk'
------------------------------------------------------------

--- Top Example 2 (Review Index: 35303) ---
Review-Level Feature Activation: 2.2242

Highlighted Review (Feature Activation Points in RED):
[[VIDEOID:a00b55bca6cdc5e2acfbb3f790d403fc]] I wntd to wait a little bit to actually use it to make my review.Came on time 

In [39]:
inspect_sae_feature(feature_idx=498, lm=lm, sae=sae, F_va=df_train_valid[[c for c in df_train_valid.columns if c.startswith('sae_')]].values, texts_va=df_train_valid['text'].values, order=order, n_reviews_to_show=10)


Inspecting SAE Feature: 498

--- Top Example 1 (Review Index: 51933) ---
Review-Level Feature Activation: 2.0251

Highlighted Review (Feature Activation Points in RED):
My dryer was older than dirt and finally died. I didn’t want to spend huge amounts of money on a new dryer right now and it’s just me in the household. So I took a chance after tons of research and went with this portable compact dryer.<br />It works great. But don’t overload it. The instructions could’ve been written better but I think personal experimentation works best to find out what settings to use. First go round hardly dried the clothes and I did get worried I would have to send back. However, after trying a few different settings I found the right one and the clothes dried nicely! They were toasty warm and nice and dry with no damp spots.<br />This will definitely work for me until sometime in the future when I decide to drop a grand for a new dryer. W who knows…..I may just stick with this and see how many ye

### lower demand


In [33]:
inspect_sae_feature(feature_idx=6, lm=lm, sae=sae, F_va=df_train_valid[[c for c in df_train_valid.columns if c.startswith('sae_')]].values, texts_va=df_train_valid['text'].values, order=order, n_reviews_to_show=10)


Inspecting SAE Feature: 6

--- Top Example 1 (Review Index: 39023) ---
Review-Level Feature Activation: 1.5414

Highlighted Review (Feature Activation Points in RED):
I got this to replace a burnt element for my Whirlpool RF366PXGW0.  This says it was compatible with the W10823696 element even though it's designed slightly different.  And before anyone says anything, yes, I do realize that connectors are placed differently on this element than the original, and yes, I did adjust the wires accordingly.  Except it didn't work.  So I don't know if I got a dud or this is just simply incompatible, because I bought a different one on Ebay but way more similar than this one, and it worked without a hitch, so it definitely wasn't my range oven, nor the wirings, but this piece.  From the reviews, some managed to get this to work with theirs even if it was slightly different, so I hope it will with yours as well.  For me, it did not.  Therefore I can't recommend it.


Activating Spans Found:
  

In [28]:
inspect_sae_feature(feature_idx=1, lm=lm, sae=sae, F_va=df_train_valid[[c for c in df_train_valid.columns if c.startswith('sae_')]].values, texts_va=df_train_valid['text'].values, order=order, n_reviews_to_show=10)


Inspecting SAE Feature: 1

--- Top Example 1 (Review Index: 2015) ---
Review-Level Feature Activation: 2.2074

Highlighted Review (Feature Activation Points in RED):
What I thought was a seized motor bearing turned out quite differently. I could not remove the fan blade, and ended up breaking the square spacer end off the blade. Poor cheap design on the fan blade. So now I had no choice but to open the rear of the dryer ( no big deal ) and remove the fan blade. I took off the vent assembly and to my amazement found a small rabbit wedged between the fan blade and the rear housing. Neither the fan blade or the rabbit were going anywhere. The original motor is fine, but I now have a spare. I MacGyver'd a fix for blade rub problem caused by breaking the square spacer that buts up to the rear of the motor. Opening up the vent system gave me the opportunity to thoroughly delint the vent system. I also added a rodent cage over the vent to keep out unsuspecting bunnies and rodents. Thankfully

In [32]:
inspect_sae_feature(feature_idx=2294, lm=lm, sae=sae, F_va=df_train_valid[[c for c in df_train_valid.columns if c.startswith('sae_')]].values, texts_va=df_train_valid['text'].values, order=order, n_reviews_to_show=10)


Inspecting SAE Feature: 2294

--- Top Example 1 (Review Index: 19450) ---
Review-Level Feature Activation: 1.9662

Highlighted Review (Feature Activation Points in RED):
Works great!! I got the 688 for 18$ inside the box deal which I didn't need. Being that the Broan spun CCW I had to flip the magnet on it and now it spins CW. Thus this CW fan works and sucks air up and out. I originally had a Bojack generic bathroom fan which had a whine that got to 79db's! It would quiet down after a bit, but it wasn't acceptable. So I set out to find a better option. Knowing that Broan is actually a decent brand I decided to piece one together since from what I could find there wasn't a stop setup with a 6.5"ish blade. That fit onto a shaft of the Broan motor(like 4.6mm's). In the end it works great and has ZERO electrical whine. You can hear air moving but not anything that's unreasonable.


Activating Spans Found:
  - (Score: 2.1875) @ [278:281]: ' got'
-------------------------------------------

In [13]:
inspect_sae_feature(feature_idx=0, lm=lm, sae=sae, F_va=df_train_valid[[c for c in df_train_valid.columns if c.startswith('sae_')]].values, texts_va=df_train_valid['text'].values, order=order, n_reviews_to_show=10)


Inspecting SAE Feature: 0

--- Top Example 1 (Review Index: 48532) ---
Review-Level Feature Activation: 0.7122

Highlighted Review (Feature Activation Points in RED):
This ice machine is not worth the money. Stopped working shortly after installation. Reached out the manufacturer and they replaced it with a refurbished machine, that also doesn't work and now they won't respond to emails. BAD customer service and product!


Activating Spans Found:
  - (Score: 0.7123) @ [83:84]: '.'
------------------------------------------------------------

--- Top Example 2 (Review Index: 80197) ---
Review-Level Feature Activation: 0.7059

Highlighted Review (Feature Activation Points in RED):
The Evanro 30" Slide-in Range Rear Filler Kit is well made and looks nice. It fits like it was made for my Jenn-air stove. With that said mine did not come with any mounting instructions, but it is clear how the brackets attach. Without instructions I cannot be sure but it certainly appears that the brackets c

In [14]:
inspect_sae_feature(feature_idx=1119, lm=lm, sae=sae, F_va=df_train_valid[[c for c in df_train_valid.columns if c.startswith('sae_')]].values, texts_va=df_train_valid['text'].values, order=order, n_reviews_to_show=10)


Inspecting SAE Feature: 1119

--- Top Example 1 (Review Index: 80657) ---
Review-Level Feature Activation: 0.3162

Highlighted Review (Feature Activation Points in RED):
One of the biggest problems with making my own espresso has always been cleaning up afterwards. Grounds always end up working their way inside the machine, and it takes tools to truly ensure it's all out. I'd been using the same machine since before Amazon was even around, but recently replaced it and decided to see what was new in accessories, too.<br /><br />My new machine came with few extras, just a drip tray, couple different insert sizes for the handset, and a scoop/tamper. Before, I would've thought that's all you really needed. I was wrong.<br /><br />When I first saw one of these filters, I knew I had to have one, and I was right. A carafe, espresso cups, a milk pitcher... all those things are nice, but unnecessary for making a superb latté. This filter not only keeps the machine far cleaner, but causes more 

In [15]:
inspect_sae_feature(feature_idx=2859, lm=lm, sae=sae, F_va=df_train_valid[[c for c in df_train_valid.columns if c.startswith('sae_')]].values, texts_va=df_train_valid['text'].values, order=order, n_reviews_to_show=10)


Inspecting SAE Feature: 2859

--- Top Example 1 (Review Index: 22411) ---
Review-Level Feature Activation: 0.5211

Highlighted Review (Feature Activation Points in RED):
The package came with all (5) of the parts needed for repairing the heating side of clothes dryer. From the actual element which is what I needed, to the fuse and thermostats. Much cheaper to fix a reliable dryer (&lt;50 dollars) to buying one (500~), of which hardly any had really good reviews.


Activating Spans Found:
  - (Score: 0.6268) @ [217:218]: ';'
------------------------------------------------------------

--- Top Example 2 (Review Index: 76471) ---
Review-Level Feature Activation: 0.4990

Highlighted Review (Feature Activation Points in RED):
I was super excited about the possibility of this to bring more red accents to our kitchen & maybe even help keep the handles cleaner.  However, there are a lot of drawbacks to these:<br /><br />1)  The material isn't stretchy or flexible enough to easily accommodate

In [16]:
inspect_sae_feature(feature_idx=1193, lm=lm, sae=sae, F_va=df_train_valid[[c for c in df_train_valid.columns if c.startswith('sae_')]].values, texts_va=df_train_valid['text'].values, order=order, n_reviews_to_show=10)


Inspecting SAE Feature: 1193

--- Top Example 1 (Review Index: 8251) ---
Review-Level Feature Activation: 2.2792

Highlighted Review (Feature Activation Points in RED):
Me encanta y me hace la vida más fácil; vivo en un apartamento y para lavar tenía que bajar las escaleras hacia el sótano .. Era estresante cargar montónes de canastas 🧺 de ropa sucia 😰. Estaba indecisa 😕 porque tanto hay buenos comentarios como también los hay malos ...pero me arriesgué y aquí me tienes poniendo mi opinión. Hasta el momento es muy buena .,mejor de lo imaginado.. volveré por si noto algo mal pero llevo 3 meses y es de lo mejor...ahora tengo mi pequeña lavandería en mi apartamento... (lavadora portátil y secadora portátil)😍🥰🥰🥰feliz


Activating Spans Found:
  - (Score: 2.6510) @ [361:362]: 'j'
  - (Score: 2.1842) @ [509:511]: 'ad'
------------------------------------------------------------

--- Top Example 2 (Review Index: 64693) ---
Review-Level Feature Activation: 1.9596

Highlighted Review (Feature 